# Skalowanie cech / standaryzacja — kompletny przewodnik

**Problem:** "skalowanie danych" to nie jedna operacja, tylko rodzina metod o różnych założeniach — mylenie ich (np. `Normalizer` z `StandardScaler`) albo stosowanie niewłaściwej metody do rozkładu danych (np. `MinMaxScaler` na danych z outlierem) daje wynik, który wygląda poprawnie, ale psuje dalszą analizę/model po cichu.

**Kiedy w ogóle skalować:**
- **Modele oparte na odległości lub gradiencie** (KNN, SVM, K-means, PCA, sieci neuronowe, regresja z regularyzacją — Ridge/Lasso) — skalowanie ZMIENIA wynik, nie tylko jego interpretację. Bez niego kolumna o większych wartościach bezwzględnych (np. dochód w tysiącach) zdominuje kolumnę o mniejszych (np. wiek) tylko dlatego, że ma większą skalę liczbową, nie większe znaczenie.
- **Modele drzewiaste** (drzewa decyzyjne, Random Forest, XGBoost/LightGBM) — **nie wymagają skalowania w ogóle**. Zweryfikowane niżej: drzewo daje identyczne predykcje na surowych i standaryzowanych danych, bo dzieli dane po progach wartości, niezależnie od skali.
- **Zwykła (nieregularyzowana) regresja liniowa** — predykcje wychodzą identyczne ze skalowaniem i bez, ale WSPÓŁCZYNNIKI się zmieniają — skalowanie tu wpływa na interpretowalność, nie na jakość predykcji.

**Porównanie metod w skrócie:** `StandardScaler` (średnia/odch. std) dla danych w miarę symetrycznych; `RobustScaler` (mediana/IQR) gdy są outliery; `MinMaxScaler` gdy potrzebny konkretny zakres (np. `[0,1]`) i outlierów brak; `MaxAbsScaler` dla danych rzadkich/nieujemnych; `Normalizer` to zupełnie inna oś (normalizacja WIERSZOWA, nie kolumnowa!); `PowerTransformer`/`QuantileTransformer`/`log` do redukcji skośności, nie tylko zmiany skali.

## Setup

Realistyczny zbiór klientów: `income` ma naturalny prawoskośny rozkład (typowy dla dochodów) plus jeden ekstremalny outlier (np. błąd danych albo prawdziwy nietypowy klient) — dobra baza do pokazania różnic między metodami.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
from scipy import stats

rng = np.random.default_rng(11)
n = 200

df = pd.DataFrame({
    "age": rng.integers(20, 65, n),
    "income": rng.lognormal(mean=8.5, sigma=0.4, size=n).round(0),
    "tenure_days": rng.integers(30, 3000, n),
    "purchases": rng.poisson(5, n),
})
df.loc[0, "income"] = 900_000  # ekstremalny outlier

print(f"Skośność income: {df['income'].skew():.2f} (mocno prawoskośne)")
df.describe()

## Sekcja 0 — Dowód: drzewa decyzyjne NIE potrzebują skalowania

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

X = df[["age", "income"]]
y = X["age"] * 10 + X["income"] * 0.01 + rng.normal(0, 50, n)

X_scaled = StandardScaler().fit_transform(X)

tree_raw = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X, y)
tree_scaled = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_scaled, y)

print(f"Drzewo: predykcje identyczne surowe vs standaryzowane? {np.allclose(tree_raw.predict(X), tree_scaled.predict(X_scaled))}")

lr_raw = LinearRegression().fit(X, y)
lr_scaled = LinearRegression().fit(X_scaled, y)
print(f"\nRegresja liniowa: predykcje identyczne? {np.allclose(lr_raw.predict(X), lr_scaled.predict(X_scaled))}")
print(f"Ale współczynniki RAW:    {lr_raw.coef_}")
print(f"Współczynniki SCALED: {lr_scaled.coef_}")

## Sekcja 1 — `StandardScaler` (standaryzacja Z-score)

$(x - \text{średnia}) / \text{odch. std}$ — wynik ma średnią 0 i odchylenie standardowe 1. Klasyczny wybór, gdy dane są w miarę symetryczne i model zakłada rozkład zbliżony do normalnego (regresja logistyczna, SVM, PCA, sieci neuronowe). **Wrażliwy na outliery** — patrz test niżej.

In [ ]:
scaler = StandardScaler()
income_scaled = scaler.fit_transform(df[["income"]])

print(f"Mean: {scaler.mean_[0]:.1f}, Std: {scaler.scale_[0]:.1f}")
print(f"Pierwsze 5 wartości (0. to nasz outlier): {income_scaled[:5].ravel().round(2)}")
print(f"Ile z 200 obserwacji mieści się w [-1, 1]: {((income_scaled > -1) & (income_scaled < 1)).sum()}")
print("-> jeden ekstremalny outlier tak bardzo napompował odchylenie standardowe,")
print("   że niemal WSZYSTKIE pozostałe, normalne wartości wyglądają na 'blisko średniej'")

### To samo ręcznie w pandas/polars — i pułapka `ddof`

`pandas.std()` domyślnie dzieli przez `n-1` (odchylenie standardowe **próby**, `ddof=1`) — `sklearn`/`numpy` domyślnie dzielą przez `n` (odchylenie **populacji**, `ddof=0`). Ręczna replikacja `StandardScaler` w czystym pandas bez `ddof=0` da INNY wynik niż sklearn — niewielki, ale realny.

In [ ]:
print(f"pandas .std() (ddof=1, domyślne):  {df['income'].std():.2f}")
print(f"numpy .std()  (ddof=0, domyślne):  {df['income'].to_numpy().std():.2f}")
print(f"sklearn StandardScaler (ddof=0):   {scaler.scale_[0]:.2f}")
print("\nAby ręcznie odtworzyć sklearn w pandas, trzeba jawnie: .std(ddof=0)")

# Poprawna replikacja w pandas
z_pandas = (df["income"] - df["income"].mean()) / df["income"].std(ddof=0)
print(f"Zgodność z sklearn: {np.allclose(z_pandas, income_scaled.ravel())}")

In [ ]:
# polars .std() domyślnie TAKŻE używa ddof=1 (jak pandas) - ta sama zasada obowiązuje
df_pl = pl.from_pandas(df[["income"]])
z_polars = df_pl.with_columns(
    ((pl.col("income") - pl.col("income").mean()) / pl.col("income").std(ddof=0)).alias("income_z")
)
z_polars.head(3)

### `scipy.stats.zscore` — to samo, jako jedna funkcja bez `fit`/`transform`

Wygodne do szybkiej eksploracji, gdy nie potrzebujesz zapamiętać parametrów skalowania do późniejszego użycia na nowych danych (patrz Pułapka 1 — tam `fit`/`transform` sklearn ma realną przewagę).

In [ ]:
z_scipy = stats.zscore(df["income"])
print(f"Mean: {z_scipy.mean():.6f}, Std: {z_scipy.std():.4f}")

## Sekcja 2 — `MinMaxScaler`

Przeskalowanie do konkretnego zakresu (domyślnie `[0, 1]`). Przydatne, gdy model/algorytm wymaga wartości w ograniczonym przedziale (np. niektóre sieci neuronowe, wizualizacje). **Bardzo wrażliwy na outliery** — jeszcze bardziej niż `StandardScaler`.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
income_mm = minmax.fit_transform(df[["income"]])

print(f"Zakres oryginalny: {df['income'].min():.0f} - {df['income'].max():.0f}")
print(f"Percentyle 5/50/95 PO MinMax: {np.percentile(income_mm, [5, 50, 95]).round(4)}")
print("-> jeden outlier definiuje wartość 1.0, więc 95% normalnych obserwacji ląduje poniżej 0.01")

## Sekcja 3 — `RobustScaler`

$(x - \text{mediana}) / \text{IQR}$ — używa mediany i rozstępu międzykwartylowego zamiast średniej/odch. std, więc **pojedynczy outlier nie wpływa na skalowanie pozostałych obserwacji** (choć sam outlier nadal będzie miał ekstremalną wartość po transformacji — `RobustScaler` nie usuwa outlierów, tylko nie pozwala im zdominować skali dla reszty danych).

In [ ]:
from sklearn.preprocessing import RobustScaler

robust = RobustScaler()
income_robust = robust.fit_transform(df[["income"]])

print(f"Mediana: {robust.center_[0]:.1f}, IQR: {robust.scale_[0]:.1f}")
print(f"Pierwsze 5 wartości (0. to outlier): {income_robust[:5].ravel().round(2)}")
print("-> normalne obserwacje mieszczą się w rozsądnym zakresie (~-0.6 do ~0.3),")
print("   w przeciwieństwie do StandardScaler, gdzie były sztucznie 'ściśnięte' blisko zera")

## Sekcja 4 — `MaxAbsScaler`

Dzieli przez maksymalną wartość BEZWZGLĘDNĄ — **nie centruje danych** (nie odejmuje średniej/mediany). Zaleta: zachowuje strukturę zer w danych rzadkich (`sparse`) — centrowanie zamieniłoby zera na małe wartości niezerowe, niszcząc rzadkość macierzy. Typowe zastosowanie: cechy TF-IDF z przetwarzania tekstu, dane finansowe, gdzie znak (dodatni/ujemny) niesie znaczenie.

In [ ]:
from sklearn.preprocessing import MaxAbsScaler

maxabs = MaxAbsScaler()
income_maxabs = maxabs.fit_transform(df[["income"]])

print(f"max_abs_: {maxabs.max_abs_[0]:.0f}")
print(f"Zakres po skalowaniu: [{income_maxabs.min():.4f}, {income_maxabs.max():.4f}]")
print("-> BEZ centrowania: zera w danych źródłowych zostają zerami po transformacji")

## Sekcja 5 — `Normalizer`: zupełnie inna oś — normalizacja WIERSZOWA, nie kolumnowa

**To jest najczęściej mylona metoda w tej rodzinie.** Wszystkie poprzednie skalery działają NA KOLUMNIE — każda cecha jest przetwarzana niezależnie względem swoich własnych statystyk. `Normalizer` działa NA WIERSZU — każda OBSERWACJA (nie kolumna!) zostaje przeskalowana tak, żeby jej wektor cech miał jednostkową normę. Używane tam, gdzie liczy się KIERUNEK wektora cech, nie jego długość (np. podobieństwo kosinusowe w wyszukiwaniu tekstowym), **nie** jako zamiennik `StandardScaler`.

In [ ]:
from sklearn.preprocessing import Normalizer

sample = df[["age", "tenure_days", "purchases"]].head(3)
print("Przed:")
print(sample)

normalized = Normalizer(norm="l2").fit_transform(sample)
print("\nPo Normalizer (każdy WIERSZ ma normę euklidesową = 1):")
print(normalized.round(4))
print(f"\nSuma kwadratów każdego wiersza (musi = 1): {(normalized**2).sum(axis=1).round(4)}")

**Dlaczego to niebezpieczne, jeśli pomylone ze `StandardScaler`:** powyżej `tenure_days` (rząd tysięcy) całkowicie zdominował kierunek każdego wektora (~0.999), a `age` i `purchases` (rząd dziesiątek/jedności) praktycznie znikły (~0.02–0.05) — mimo że merytorycznie mogą być równie ważne. `Normalizer` bez wcześniejszego wyrównania skali kolumn nie "naprawia" różnic w jednostkach — wręcz przeciwnie, jeszcze mocniej faworyzuje kolumnę o największych wartościach bezwzględnych.

## Sekcja 6 — `PowerTransformer`: Box-Cox i Yeo-Johnson

To nie jest zwykłe skalowanie — to transformacja ZMNIEJSZAJĄCA SKOŚNOŚĆ, robiąca rozkład bardziej zbliżonym do normalnego. Przydatne PRZED `StandardScaler`, jeśli model zakłada normalność cech (regresja liniowa z istotnością statystyczną, niektóre testy statystyczne).

- **Box-Cox** — wymaga danych ŚCIŚLE dodatnich (`> 0`).
- **Yeo-Johnson** — działa też z zerami i wartościami ujemnymi (rozszerzenie Box-Coxa).

In [ ]:
from sklearn.preprocessing import PowerTransformer

pt_boxcox = PowerTransformer(method="box-cox")
income_bc = pt_boxcox.fit_transform(df[["income"]])

print(f"Skośność PRZED:        {df['income'].skew():.2f}")
print(f"Skośność PO Box-Cox:   {pd.Series(income_bc.ravel()).skew():.2f}")
print(f"Dobrana automatycznie lambda: {pt_boxcox.lambdas_[0]:.3f}")

### Box-Cox rzuca błąd na danych z zerem — Yeo-Johnson sobie radzi

In [ ]:
data_with_zero = np.array([0, 5, 10, 15, 20])

try:
    stats.boxcox(data_with_zero)
except ValueError as e:
    print(f"Box-Cox na danych z zerem -> błąd: {e}")

yj_result = PowerTransformer(method="yeo-johnson").fit_transform(data_with_zero.reshape(-1, 1))
print(f"\nYeo-Johnson na tych samych danych: {yj_result.ravel().round(3)}")

## Sekcja 7 — `QuantileTransformer`

Najbardziej agresywna, nieliniowa transformacja w tym zestawie — mapuje dane na rozkład jednostajny (`uniform`) albo normalny (`normal`) na podstawie ich RANG (kolejności), nie wartości bezwzględnych. Skrajnie odporna na outliery (bo liczy się tylko pozycja w sortowaniu), ale **potrafi zniekształcić relacje między zmiennymi** — dwie wartości blisko siebie w oryginalnych danych mogą po transformacji wylądować daleko od siebie, jeśli w tym miejscu rozkładu jest dużo obserwacji.

In [ ]:
from sklearn.preprocessing import QuantileTransformer

qt_uniform = QuantileTransformer(output_distribution="uniform", n_quantiles=100, random_state=0)
income_qu = qt_uniform.fit_transform(df[["income"]])
print(f"Zakres po transformacji: [{income_qu.min():.2f}, {income_qu.max():.2f}]")
print(f"Percentyle 25/50/75 (z definicji ~0.25/0.5/0.75): {np.percentile(income_qu, [25, 50, 75]).round(3)}")

qt_normal = QuantileTransformer(output_distribution="normal", n_quantiles=100, random_state=0)
income_qn = qt_normal.fit_transform(df[["income"]])
print(f"\nSkośność po QuantileTransformer(normal): {pd.Series(income_qn.ravel()).skew():.4f} (praktycznie 0)")

## Sekcja 8 — Prosty `log1p`: najprostsza redukcja skośności

Zanim sięgniesz po `PowerTransformer`/`QuantileTransformer`, sprawdź, czy zwykły logarytm nie wystarczy — jest prostszy do wytłumaczenia ("analizujemy dane w skali logarytmicznej", częste w finansach/ekonomii) i w pełni odwracalny (`np.expm1`). `log1p` (log(1+x)) zamiast zwykłego `log`, żeby bezpiecznie obsłużyć zera.

In [ ]:
log_income = np.log1p(df["income"])
print(f"Skośność PRZED:      {df['income'].skew():.2f}")
print(f"Skośność PO log1p:   {log_income.skew():.2f}")
print("-> lepiej niż nic, ale mniej skuteczne niż Box-Cox/QuantileTransformer (mniej agresywne, bo nie dopasowuje parametru do danych)")

## Sekcja 9 — Pułapki

### Pułapka 1 — `fit` na CAŁYM zbiorze przed podziałem train/test = wyciek danych

Skaler musi poznać średnią/odchylenie/min/max WYŁĄCZNIE ze zbioru treningowego. Jeśli `fit()` widzi też dane testowe, statystyki skalowania "wiedzą" coś o zbiorze, którego model formalnie jeszcze nie widział — klasyczny data leakage, zaniżający realną ocenę jakości modelu.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(df[["income"]], test_size=0.2, random_state=42)

# BŁĘDNIE: fit na całym df (widzi też przyszły zbiór testowy)
scaler_leaky = StandardScaler().fit(df[["income"]])

# POPRAWNIE: fit TYLKO na treningowym
scaler_correct = StandardScaler().fit(X_train)

print(f"Mean z pełnego zbioru (wyciek):     {scaler_leaky.mean_[0]:.2f}")
print(f"Mean z samego treningu (poprawnie): {scaler_correct.mean_[0]:.2f}")
print("\nRóżnica bywa niewielka na losowym podziale, ale rośnie przy mniejszych zbiorach")
print("albo bardziej skośnych/niezbalansowanych danych — zasada obowiązuje zawsze, niezależnie od wielkości efektu.")

X_test_scaled = scaler_correct.transform(X_test)  # ten sam scaler, dopasowany tylko do treningu

### Powiązane: `inverse_transform` — powrót do oryginalnej skali

Jeśli model przewiduje wartość na przeskalowanych danych (np. cenę po standaryzacji), wynik trzeba jawnie odwrócić przed prezentacją użytkownikowi końcowemu.

In [ ]:
X_train_scaled = scaler_correct.transform(X_train)
back_to_original = scaler_correct.inverse_transform(X_train_scaled)

print(f"inverse_transform odtwarza oryginał: {np.allclose(back_to_original, X_train.values)}")

### Pułapka 2 — regularyzowane modele liniowe (Ridge/Lasso), KNN, SVM, PCA, K-means WYMAGAJĄ skalowania

Sekcja 0 pokazała, że zwykła regresja liniowa daje identyczne PREDYKCJE niezależnie od skalowania. To nie dotyczy modeli z regularyzacją (`Ridge`, `Lasso`) — kara nakładana na współczynniki jest wrażliwa na ich skalę, więc kolumna o dużych wartościach dostanie sztucznie mniejszą karę względem kolumny o małych wartościach, jeśli dane nie są ujednolicone. To samo dotyczy KNN/SVM/K-means (odległość euklidesowa dominowana przez kolumnę o największej skali) i PCA (wariancja, na której bazuje metoda, też jest skalowo zależna).

## Podsumowanie: kiedy która metoda

| Metoda | Wzór/mechanizm | Kiedy stosować | Kiedy NIE stosować |
|---|---|---|---|
| `StandardScaler` | $(x-\bar{x})/\sigma$ | Dane w miarę symetryczne, model zakłada rozkład zbliżony do normalnego | Silne outliery — zdominują $\sigma$ |
| `MinMaxScaler` | $(x-\min)/(\max-\min)$ | Potrzebny konkretny, ograniczony zakres (np. wejście sieci neuronowej) | Outliery — jeden ekstremalny punkt ściska resztę blisko 0 |
| `RobustScaler` | $(x-\text{mediana})/\text{IQR}$ | Dane z outlierami, których nie chcesz usuwać | Gdy i tak nie ma outlierów — `StandardScaler` jest prostszy koncepcyjnie |
| `MaxAbsScaler` | $x/\max\lvert x\rvert$ | Dane rzadkie (sparse), zachowanie zer, dane już wyśrodkowane wokół 0 | Dane silnie asymetryczne bez wcześniejszego wyśrodkowania |
| `Normalizer` | Wiersz / norma wiersza | Kierunek wektora cech ważniejszy niż jego długość (np. podobieństwo kosinusowe) | **Nigdy jako zamiennik pozostałych** — to inna oś (wiersze, nie kolumny) |
| `PowerTransformer` (Box-Cox) | Optymalizowana transformacja potęgowa | Redukcja skośności, dane ściśle dodatnie | Dane z zerem/ujemnymi — użyj Yeo-Johnson |
| `PowerTransformer` (Yeo-Johnson) | Jw., rozszerzone | Jw., ale dane mogą zawierać 0/wartości ujemne | — |
| `QuantileTransformer` | Mapowanie wg rangi/percentyla | Bardzo nieregularne rozkłady, silna odporność na outliery potrzebna za wszelką cenę | Gdy zależy Ci na zachowaniu relacji odległości między wartościami |
| `log1p` | $\log(1+x)$ | Szybka, prosta, interpretowalna redukcja skośności (dane finansowe) | Gdy potrzebna precyzyjna normalizacja — słabsza niż Box-Cox |
| Brak skalowania | — | Modele drzewiaste (drzewa, Random Forest, XGBoost/LightGBM) | — |

**Wniosek:** wybór metody skalowania to decyzja o tym, JAK rozkład danych ma wpłynąć na model — nie ma jednej "domyślnie bezpiecznej" opcji. `StandardScaler` jako uniwersalny domyślny wybór zawodzi dokładnie tam, gdzie dane realnie mają outliery (czyli dość często w danych biznesowych) — `RobustScaler` bywa bezpieczniejszym punktem startowym, jeśli nie masz pewności co do rozkładu.